# CIF文件全面分析

本notebook将使用biotite库来深入分析CIF文件的所有组成部分，包括：
- 元数据信息
- 结构信息
- 原子坐标
- 实验数据
- 细化参数
- 其他注释信息

In [3]:
import biotite.structure.io.pdbx as pdbx
import biotite.structure as struc
import numpy as np
import pandas as pd
from pprint import pprint
import json

# 读取CIF文件
cif_file = pdbx.CIFFile.read("1qkt.cif")
print("成功读取CIF文件")
print(f"数据块数量: {len(cif_file)}")
print(f"数据块名称: {list(cif_file.keys())}")

成功读取CIF文件
数据块数量: 1
数据块名称: ['1QKT']


In [4]:
# 获取主数据块
data_block = cif_file["1QKT"]

# 获取所有的类别（categories）
categories = list(data_block.keys())
print(f"CIF文件包含 {len(categories)} 个数据类别：")
print("="*60)

# 按字母顺序排序并显示
for i, category in enumerate(sorted(categories), 1):
    print(f"{i:2d}. {category}")
    
print("\n" + "="*60)
print(f"总计: {len(categories)} 个数据类别")

CIF文件包含 68 个数据类别：
 1. atom_site
 2. atom_sites
 3. atom_type
 4. audit_author
 5. audit_conform
 6. cell
 7. chem_comp
 8. chem_comp_atom
 9. chem_comp_bond
10. citation
11. citation_author
12. database_2
13. database_PDB_matrix
14. diffrn
15. diffrn_detector
16. diffrn_radiation
17. diffrn_radiation_wavelength
18. diffrn_source
19. entity
20. entity_poly
21. entity_poly_seq
22. entity_src_gen
23. entry
24. exptl
25. exptl_crystal
26. exptl_crystal_grow
27. pdbx_audit_revision_category
28. pdbx_audit_revision_details
29. pdbx_audit_revision_group
30. pdbx_audit_revision_history
31. pdbx_audit_revision_item
32. pdbx_database_related
33. pdbx_database_status
34. pdbx_distant_solvent_atoms
35. pdbx_entity_nonpoly
36. pdbx_entry_details
37. pdbx_nonpoly_scheme
38. pdbx_poly_seq_scheme
39. pdbx_struct_assembly
40. pdbx_struct_assembly_gen
41. pdbx_struct_assembly_prop
42. pdbx_struct_oper_list
43. pdbx_struct_sheet_hbond
44. pdbx_validate_symm_contact
45. pdbx_validate_torsion
46. pdbx_xplo

## 1. 基本条目信息 (Entry Information)

In [8]:
# 1. 基本条目信息
print("=== 条目基本信息 ===")
if "entry" in data_block:
    entry_info = data_block["entry"]
    for key, value in entry_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 数据库信息 ===")
if "database_2" in data_block:
    db_info = data_block["database_2"]
    print("数据库记录:")
    
    # 使用as_array()方法获取数据
    db_ids = db_info["database_id"].as_array()
    db_codes = db_info["database_code"].as_array()
    
    for i in range(len(db_ids)):
        print(f"  数据库: {db_ids[i]}")
        print(f"  代码: {db_codes[i]}")
        if 'pdbx_database_accession' in db_info:
            accessions = db_info['pdbx_database_accession'].as_array()
            print(f"  登录号: {accessions[i]}")
        if 'pdbx_DOI' in db_info:
            dois = db_info['pdbx_DOI'].as_array()
            print(f"  DOI: {dois[i]}")
        print()

print("=== 数据库状态 ===")
if "pdbx_database_status" in data_block:
    db_status = data_block["pdbx_database_status"]
    for key, value in db_status.items():
        print(f"  {key}: {value.as_array()}")

=== 条目基本信息 ===
  id: ['1QKT']

=== 数据库信息 ===
数据库记录:
  数据库: PDB
  代码: 1QKT
  登录号: pdb_00001qkt
  DOI: 10.2210/pdb1qkt/pdb

  数据库: PDBE
  代码: EBI-3012
  登录号: ?
  DOI: ?

  数据库: WWPDB
  代码: D_1290003012
  登录号: ?
  DOI: ?

=== 数据库状态 ===
  status_code: ['REL']
  entry_id: ['1QKT']
  deposit_site: ['PDBE']
  process_site: ['PDBE']
  SG_entry: ['.']
  recvd_initial_deposition_date: ['1999-08-05']
  pdb_format_compatible: ['Y']
  status_code_sf: ['REL']
  status_code_mr: ['?']
  status_code_cs: ['?']
  methods_development_category: ['?']
  status_code_nmr_data: ['?']


## 2. 结构基本信息 (Structure Information)

In [9]:
# 2. 结构基本信息
print("=== 结构标题和关键词 ===")
if "struct" in data_block:
    struct_info = data_block["struct"]
    for key, value in struct_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 结构关键词 ===")
if "struct_keywords" in data_block:
    keywords = data_block["struct_keywords"]
    for key, value in keywords.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 晶胞参数 ===")
if "cell" in data_block:
    cell_info = data_block["cell"]
    for key, value in cell_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 对称性信息 ===")
if "symmetry" in data_block:
    symmetry_info = data_block["symmetry"]
    for key, value in symmetry_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 实体信息 ===")
if "entity" in data_block:
    entity_info = data_block["entity"]
    print("生物实体:")
    entity_ids = entity_info["id"].as_array()
    entity_types = entity_info["type"].as_array()
    entity_descriptions = entity_info["pdbx_description"].as_array()
    
    for i in range(len(entity_ids)):
        print(f"  实体 {entity_ids[i]}:")
        print(f"    类型: {entity_types[i]}")
        print(f"    描述: {entity_descriptions[i]}")
        print()

=== 结构标题和关键词 ===
  entry_id: ['1QKT']
  title: ['MUTANT ESTROGEN NUCLEAR RECEPTOR LIGAND BINDING DOMAIN COMPLEXED WITH ESTRADIOL']
  pdbx_model_details: ['?']
  pdbx_CASP_flag: ['?']
  pdbx_model_type_details: ['?']

=== 结构关键词 ===
  entry_id: ['1QKT']
  pdbx_keywords: ['NUCLEAR RECEPTOR']
  text: ['NUCLEAR RECEPTOR, AGONISM, ANTAGONISM, STEROID, STRUCTURAL PROTEOMICS IN EUROPE, SPINE, STRUCTURAL GENOMICS']

=== 晶胞参数 ===
  entry_id: ['1QKT']
  length_a: ['58.610']
  length_b: ['58.610']
  length_c: ['276.020']
  angle_alpha: ['90.00']
  angle_beta: ['90.00']
  angle_gamma: ['120.00']
  Z_PDB: ['12']
  pdbx_unique_axis: ['?']

=== 对称性信息 ===
  entry_id: ['1QKT']
  space_group_name_H-M: ['P 65 2 2']
  pdbx_full_space_group_name_H-M: ['?']
  cell_setting: ['?']
  Int_Tables_number: ['179']

=== 实体信息 ===
生物实体:
  实体 1:
    类型: polymer
    描述: ESTRADIOL RECEPTOR

  实体 2:
    类型: non-polymer
    描述: ESTRADIOL

  实体 3:
    类型: water
    描述: water



## 3. 实验方法和数据收集 (Experimental Methods)

In [10]:
# 3. 实验方法和数据收集
print("=== 实验方法 ===")
if "exptl" in data_block:
    exptl_info = data_block["exptl"]
    for key, value in exptl_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 晶体信息 ===")
if "exptl_crystal" in data_block:
    crystal_info = data_block["exptl_crystal"]
    for key, value in crystal_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 晶体生长 ===")
if "exptl_crystal_grow" in data_block:
    grow_info = data_block["exptl_crystal_grow"]
    for key, value in grow_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 衍射实验 ===")
if "diffrn" in data_block:
    diffrn_info = data_block["diffrn"]
    for key, value in diffrn_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== X射线源 ===")
if "diffrn_source" in data_block:
    source_info = data_block["diffrn_source"]
    for key, value in source_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 辐射信息 ===")
if "diffrn_radiation" in data_block:
    radiation_info = data_block["diffrn_radiation"]
    for key, value in radiation_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 波长信息 ===")
if "diffrn_radiation_wavelength" in data_block:
    wavelength_info = data_block["diffrn_radiation_wavelength"]
    for key, value in wavelength_info.items():
        print(f"  {key}: {value.as_array()}")

=== 实验方法 ===
  entry_id: ['1QKT']
  method: ['X-RAY DIFFRACTION']
  crystals_number: ['1']

=== 晶体信息 ===
  id: ['1']
  density_meas: ['?']
  density_Matthews: ['2.42']
  density_percent_sol: ['45']
  description: ['?']

=== 晶体生长 ===
  crystal_id: ['1']
  method: ['?']
  temp: ['?']
  temp_details: ['?']
  pH: ['7.00']
  pdbx_pH_range: ['?']
  pdbx_details: ['pH 7.00']

=== 衍射实验 ===
  id: ['1']
  ambient_temp: ['110.0']
  ambient_temp_details: ['?']
  crystal_id: ['1']

=== X射线源 ===
  diffrn_id: ['1']
  source: ['SYNCHROTRON']
  type: ['LURE']
  pdbx_synchrotron_site: ['LURE']
  pdbx_synchrotron_beamline: ['?']
  pdbx_wavelength: ['0.9']
  pdbx_wavelength_list: ['?']

=== 辐射信息 ===
  diffrn_id: ['1']
  wavelength_id: ['1']
  pdbx_monochromatic_or_laue_m_l: ['M']
  monochromator: ['?']
  pdbx_diffrn_protocol: ['SINGLE WAVELENGTH']
  pdbx_scattering_type: ['x-ray']

=== 波长信息 ===
  id: ['1']
  wavelength: ['0.9']
  wt: ['1.0']


## 4. 结构细化统计 (Refinement Statistics)

In [11]:
# 4. 结构细化统计
print("=== 细化信息 ===")
if "refine" in data_block:
    refine_info = data_block["refine"]
    for key, value in refine_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 细化历史 ===")
if "refine_hist" in data_block:
    hist_info = data_block["refine_hist"]
    for key, value in hist_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 细化分析 ===")
if "refine_analyze" in data_block:
    analyze_info = data_block["refine_analyze"]
    for key, value in analyze_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 反射数据 ===")
if "reflns" in data_block:
    reflns_info = data_block["reflns"]
    for key, value in reflns_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 细化约束 ===")
if "refine_ls_restr" in data_block:
    restr_info = data_block["refine_ls_restr"]
    print("约束类型和统计:")
    restr_types = restr_info["type"].as_array()
    restr_numbers = restr_info["number"].as_array()
    
    for i in range(len(restr_types)):
        print(f"  {restr_types[i]}: {restr_numbers[i]}")

print("\n=== 分辨率壳层统计 ===")
if "refine_ls_shell" in data_block:
    shell_info = data_block["refine_ls_shell"]
    for key, value in shell_info.items():
        print(f"  {key}: {value.as_array()}")

=== 细化信息 ===
  pdbx_refine_id: ['X-RAY DIFFRACTION']
  entry_id: ['1QKT']
  pdbx_diffrn_id: ['1']
  pdbx_TLS_residual_ADP_flag: ['?']
  ls_number_reflns_obs: ['14756']
  ls_number_reflns_all: ['?']
  pdbx_ls_sigma_I: ['?']
  pdbx_ls_sigma_F: ['0.0']
  pdbx_data_cutoff_high_absF: ['2742495.12']
  pdbx_data_cutoff_low_absF: ['?']
  pdbx_data_cutoff_high_rms_absF: ['?']
  ls_d_res_low: ['15.00']
  ls_d_res_high: ['2.20']
  ls_percent_reflns_obs: ['97.0']
  ls_R_factor_obs: ['0.223']
  ls_R_factor_all: ['?']
  ls_R_factor_R_work: ['0.223']
  ls_R_factor_R_free: ['0.273']
  ls_R_factor_R_free_error: ['0.010']
  ls_R_factor_R_free_error_details: ['?']
  ls_percent_reflns_R_free: ['4.8']
  ls_number_reflns_R_free: ['715']
  ls_number_parameters: ['?']
  ls_number_restraints: ['?']
  occupancy_min: ['?']
  occupancy_max: ['?']
  correlation_coeff_Fo_to_Fc: ['?']
  correlation_coeff_Fo_to_Fc_free: ['?']
  B_iso_mean: ['34.3']
  aniso_B[1][1]: ['2.85']
  aniso_B[2][2]: ['2.85']
  aniso_B[3][3]: 

## 5. 原子坐标和结构信息 (Atomic Coordinates)

In [12]:
# 5. 原子坐标和结构信息
print("=== 原子位点统计 ===")
if "atom_site" in data_block:
    atom_site = data_block["atom_site"]
    
    # 获取原子数量
    total_atoms = len(atom_site["id"].as_array())
    print(f"总原子数: {total_atoms}")
    
    # 统计原子类型
    atom_types = atom_site["type_symbol"].as_array()
    unique_types, counts = np.unique(atom_types, return_counts=True)
    print("\n原子类型统计:")
    for atom_type, count in zip(unique_types, counts):
        print(f"  {atom_type}: {count}")
    
    # 统计残基类型
    if "label_comp_id" in atom_site:
        comp_ids = atom_site["label_comp_id"].as_array()
        unique_comps, comp_counts = np.unique(comp_ids, return_counts=True)
        print(f"\n残基类型统计 (前20个):")
        for comp_id, count in zip(unique_comps[:20], comp_counts[:20]):
            print(f"  {comp_id}: {count}")
    
    # 显示坐标范围
    if "Cartn_x" in atom_site:
        x_coords = atom_site["Cartn_x"].as_array().astype(float)
        y_coords = atom_site["Cartn_y"].as_array().astype(float)
        z_coords = atom_site["Cartn_z"].as_array().astype(float)
        
        print(f"\n坐标范围:")
        print(f"  X: {x_coords.min():.3f} 到 {x_coords.max():.3f}")
        print(f"  Y: {y_coords.min():.3f} 到 {y_coords.max():.3f}")
        print(f"  Z: {z_coords.min():.3f} 到 {z_coords.max():.3f}")
    
    # B因子统计
    if "B_iso_or_equiv" in atom_site:
        b_factors = atom_site["B_iso_or_equiv"].as_array()
        # 过滤掉非数值的B因子
        numeric_b = []
        for b in b_factors:
            try:
                numeric_b.append(float(b))
            except:
                pass
        
        if numeric_b:
            numeric_b = np.array(numeric_b)
            print(f"\nB因子统计:")
            print(f"  平均值: {numeric_b.mean():.2f}")
            print(f"  最小值: {numeric_b.min():.2f}")
            print(f"  最大值: {numeric_b.max():.2f}")

print("\n=== 原子类型信息 ===")
if "atom_type" in data_block:
    atom_type_info = data_block["atom_type"]
    for key, value in atom_type_info.items():
        print(f"  {key}: {value.as_array()}")

=== 原子位点统计 ===
总原子数: 2395

原子类型统计:
  C: 1285
  N: 339
  O: 755
  S: 16

残基类型统计 (前20个):
  ALA: 76
  ARG: 121
  ASN: 72
  ASP: 104
  CYS: 6
  EST: 20
  GLN: 81
  GLU: 135
  GLY: 40
  HIS: 130
  HOH: 395
  ILE: 96
  LEU: 384
  LYS: 99
  MET: 120
  PHE: 77
  PRO: 56
  SER: 120
  THR: 70
  TRP: 42

坐标范围:
  X: -22.066 到 32.919
  Y: 15.541 到 77.364
  Z: 130.974 到 186.960

B因子统计:
  平均值: 34.31
  最小值: 10.92
  最大值: 111.06

=== 原子类型信息 ===
  symbol: ['C' 'N' 'O' 'S']


## 6. 二级结构和生物组装 (Secondary Structure & Biological Assembly)

In [13]:
# 6. 二级结构和生物组装
print("=== 二级结构配置 ===")
if "struct_conf" in data_block:
    conf_info = data_block["struct_conf"]
    conf_types = conf_info["conf_type_id"].as_array()
    conf_ids = conf_info["id"].as_array()
    
    print("螺旋结构:")
    for i, (conf_type, conf_id) in enumerate(zip(conf_types, conf_ids)):
        if 'HELX' in conf_type:
            print(f"  {conf_id}: {conf_type}")
            if "beg_label_seq_id" in conf_info:
                beg_seq = conf_info["beg_label_seq_id"].as_array()[i]
                end_seq = conf_info["end_label_seq_id"].as_array()[i]
                chain_id = conf_info["beg_label_asym_id"].as_array()[i]
                print(f"    链 {chain_id}: 残基 {beg_seq}-{end_seq}")

print("\n=== β折叠结构 ===")
if "struct_sheet" in data_block:
    sheet_info = data_block["struct_sheet"]
    for key, value in sheet_info.items():
        print(f"  {key}: {value.as_array()}")

if "struct_sheet_range" in data_block:
    sheet_range = data_block["struct_sheet_range"]
    print("\nβ折叠范围:")
    sheet_ids = sheet_range["sheet_id"].as_array()
    strand_ids = sheet_range["id"].as_array()
    
    for i in range(len(sheet_ids)):
        print(f"  折叠 {sheet_ids[i]}, 链 {strand_ids[i]}:")
        if "beg_label_seq_id" in sheet_range:
            beg_seq = sheet_range["beg_label_seq_id"].as_array()[i]
            end_seq = sheet_range["end_label_seq_id"].as_array()[i]
            asym_id = sheet_range["beg_label_asym_id"].as_array()[i]
            print(f"    链 {asym_id}: 残基 {beg_seq}-{end_seq}")

print("\n=== 生物组装 ===")
if "pdbx_struct_assembly" in data_block:
    assembly_info = data_block["pdbx_struct_assembly"]
    for key, value in assembly_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 生物组装生成 ===")
if "pdbx_struct_assembly_gen" in data_block:
    assembly_gen = data_block["pdbx_struct_assembly_gen"]
    for key, value in assembly_gen.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 结构位点 ===")
if "struct_site" in data_block:
    site_info = data_block["struct_site"]
    for key, value in site_info.items():
        print(f"  {key}: {value.as_array()}")

=== 二级结构配置 ===
螺旋结构:
  HELX_P1: HELX_P
    链 A: 残基 8-19
  HELX_P2: HELX_P
    链 A: 残基 38-60
  HELX_P3: HELX_P
    链 A: 残基 63-67
  HELX_P4: HELX_P
    链 A: 残基 68-93
  HELX_P5: HELX_P
    链 A: 残基 109-113
  HELX_P6: HELX_P
    链 A: 残基 117-136
  HELX_P7: HELX_P
    链 A: 残基 138-153
  HELX_P8: HELX_P
    链 A: 残基 154-158
  HELX_P9: HELX_P
    链 A: 残基 162-191
  HELX_P10: HELX_P
    链 A: 残基 193-226
  HELX_P11: HELX_P
    链 A: 残基 232-242

=== β折叠结构 ===
  id: ['A']
  type: ['?']
  number_strands: ['2']
  details: ['?']

β折叠范围:
  折叠 A, 链 1:
    链 A: 残基 98-102
  折叠 A, 链 2:
    链 A: 残基 105-108

=== 生物组装 ===
  id: ['1']
  details: ['software_defined_assembly']
  method_details: ['PISA']
  oligomeric_details: ['dimeric']
  oligomeric_count: ['2']

=== 生物组装生成 ===
  assembly_id: ['1']
  oper_expression: ['1,2']
  asym_id_list: ['A,B,C']

=== 结构位点 ===
  id: ['AC1']
  pdbx_evidence_code: ['Software']
  pdbx_auth_asym_id: ['A']
  pdbx_auth_comp_id: ['EST']
  pdbx_auth_seq_id: ['600']
  pdbx_auth_ins_code: 

## 7. 文献和作者信息 (Citation & Authors)

In [15]:
# 7. 文献和作者信息
print("=== 文献引用 ===")
if "citation" in data_block:
    citation_info = data_block["citation"]
    for key, value in citation_info.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 文献作者 ===")
if "citation_author" in data_block:
    author_info = data_block["citation_author"]
    citation_ids = author_info["citation_id"].as_array()
    author_names = author_info["name"].as_array()
    
    # 按引用ID分组显示作者
    unique_citations = np.unique(citation_ids)
    for citation_id in unique_citations:
        print(f"\n引用 {citation_id} 的作者:")
        for i, (cid, name) in enumerate(zip(citation_ids, author_names)):
            if cid == citation_id:
                ordinal = author_info["ordinal"].as_array()[i]
                print(f"  {ordinal}. {name}")

print("\n=== 审计作者 ===")
if "audit_author" in data_block:
    audit_authors = data_block["audit_author"]
    for key, value in audit_authors.items():
        print(f"  {key}: {value.as_array()}")

print("\n=== 软件信息 ===")
if "software" in data_block:
    software_info = data_block["software"]
    print("使用的软件:")
    
    # 检查可用的字段
    available_fields = list(software_info.keys())
    print(f"可用字段: {available_fields}")
    
    software_names = software_info["name"].as_array()
    software_versions = software_info["version"].as_array()
    
    for i, (name, version) in enumerate(zip(software_names, software_versions)):
        print(f"  {name} {version}")
        
    # 显示所有软件信息字段
    print("\n详细软件信息:")
    for key, value in software_info.items():
        print(f"  {key}: {value.as_array()}")

=== 文献引用 ===
  id: ['primary']
  title: ['Crystal Structure of a Mutant Heralpha Ligand- Binding Domain Reveals Key Structural Features for the Mechanism of Partial Agonism']
  journal_abbrev: ['J.Biol.Chem.']
  journal_volume: ['276']
  page_first: ['15059']
  page_last: ['?']
  year: ['2001']
  journal_id_ASTM: ['JBCHA3']
  country: ['US']
  journal_id_ISSN: ['0021-9258']
  journal_id_CSD: ['0071']
  book_publisher: ['?']
  pdbx_database_id_PubMed: ['11278577']
  pdbx_database_id_DOI: ['10.1074/JBC.M009870200']

=== 文献作者 ===

引用 primary 的作者:
  1. Gangloff, M.
  2. Ruff, M.
  3. Eiler, S.
  4. Duclaud, S.
  5. Wurtz, J.M.
  6. Moras, D.

=== 审计作者 ===
  name: ['Ruff, M.' 'Gangloff, M.' 'Eiler, S.' 'Duclaud, S.' 'Wurtz, J.M.'
 'Moras, D.']
  pdbx_ordinal: ['1' '2' '3' '4' '5' '6']

=== 软件信息 ===
使用的软件:
可用字段: ['name', 'classification', 'version', 'citation_id', 'pdbx_ordinal']
  CNS 0.4
  DENZO .
  SCALEPACK .
  CNS 0.4

详细软件信息:
  name: ['CNS' 'DENZO' 'SCALEPACK' 'CNS']
  classification: 

## 8. CIF文件信息总结

In [16]:
# 8. CIF文件信息总结
print("="*70)
print("              CIF文件 (1QKT) 完整信息总结")
print("="*70)

print("\n📋 基本信息:")
print("  • PDB ID: 1QKT")
print("  • 标题: 突变体雌激素核受体配体结合域与雌二醇复合物")
print("  • 分类: 核受体")
print("  • 沉积日期: 1999-08-05")
print("  • 发布状态: 已发布 (REL)")

print("\n🔬 实验信息:")
print("  • 方法: X射线衍射")
print("  • 分辨率: 2.20 Å (15.00-2.20 Å)")
print("  • 空间群: P 65 2 2")
print("  • 晶胞参数: a=58.61 Å, b=58.61 Å, c=276.02 Å")
print("  • 角度: α=90°, β=90°, γ=120°")
print("  • X射线源: 同步辐射 (LURE)")
print("  • 波长: 0.9 Å")

print("\n📊 细化统计:")
print("  • R值: 22.3% (工作集)")
print("  • R-free: 27.3%")
print("  • 观测反射数: 14,756")
print("  • 平均B因子: 34.3 Ų")
print("  • 完整性: 97.0%")

print("\n🧬 结构组成:")
print("  • 总原子数: 2,395")
print("  • 蛋白质原子: 1,980")
print("  • 配体原子: 20 (雌二醇)")
print("  • 溶剂分子: 395 (水)")
print("  • 原子类型: C(1285), O(755), N(339), S(16)")

print("\n🏗️ 生物实体:")
print("  • 实体1: 聚合物 - 雌二醇受体")
print("  • 实体2: 非聚合物 - 雌二醇")
print("  • 实体3: 水分子")

print("\n🌀 二级结构:")
print("  • α螺旋: 11个 (残基8-242范围)")
print("  • β折叠: 1个 (2条链)")
print("  • 活性位点: 1个 (雌二醇结合位点)")

print("\n📚 发表信息:")
print("  • 期刊: J.Biol.Chem.")
print("  • 年份: 2001")
print("  • 卷期: 276:15059")
print("  • PubMed ID: 11278577")
print("  • DOI: 10.1074/JBC.M009870200")

print("\n👥 作者:")
print("  • Gangloff, M.; Ruff, M.; Eiler, S.; Duclaud, S.; Wurtz, J.M.; Moras, D.")

print("\n💻 使用软件:")
print("  • 数据处理: DENZO, SCALEPACK")
print("  • 结构细化: CNS 0.4")
print("  • 相位确定: CNS 0.4")

print("\n📈 质量指标:")
print("  • Matthews系数: 2.42")
print("  • 溶剂含量: 45%")
print("  • Wilson B因子: 17.0 Ų")
print("  • Rsym: 5.0%")

print("\n" + "="*70)
print("CIF文件包含了蛋白质结构的完整信息，从实验条件到最终结构的详细描述")
print("="*70)

              CIF文件 (1QKT) 完整信息总结

📋 基本信息:
  • PDB ID: 1QKT
  • 标题: 突变体雌激素核受体配体结合域与雌二醇复合物
  • 分类: 核受体
  • 沉积日期: 1999-08-05
  • 发布状态: 已发布 (REL)

🔬 实验信息:
  • 方法: X射线衍射
  • 分辨率: 2.20 Å (15.00-2.20 Å)
  • 空间群: P 65 2 2
  • 晶胞参数: a=58.61 Å, b=58.61 Å, c=276.02 Å
  • 角度: α=90°, β=90°, γ=120°
  • X射线源: 同步辐射 (LURE)
  • 波长: 0.9 Å

📊 细化统计:
  • R值: 22.3% (工作集)
  • R-free: 27.3%
  • 观测反射数: 14,756
  • 平均B因子: 34.3 Ų
  • 完整性: 97.0%

🧬 结构组成:
  • 总原子数: 2,395
  • 蛋白质原子: 1,980
  • 配体原子: 20 (雌二醇)
  • 溶剂分子: 395 (水)
  • 原子类型: C(1285), O(755), N(339), S(16)

🏗️ 生物实体:
  • 实体1: 聚合物 - 雌二醇受体
  • 实体2: 非聚合物 - 雌二醇
  • 实体3: 水分子

🌀 二级结构:
  • α螺旋: 11个 (残基8-242范围)
  • β折叠: 1个 (2条链)
  • 活性位点: 1个 (雌二醇结合位点)

📚 发表信息:
  • 期刊: J.Biol.Chem.
  • 年份: 2001
  • 卷期: 276:15059
  • PubMed ID: 11278577
  • DOI: 10.1074/JBC.M009870200

👥 作者:
  • Gangloff, M.; Ruff, M.; Eiler, S.; Duclaud, S.; Wurtz, J.M.; Moras, D.

💻 使用软件:
  • 数据处理: DENZO, SCALEPACK
  • 结构细化: CNS 0.4
  • 相位确定: CNS 0.4

📈 质量指标:
  • Matthews系数: 2.42
  • 溶剂含量: 45%
  • W

## 9. 直接查看CIF文件中特定类别的位置

In [ ]:
# 直接查看CIF文件中的特定类别位置
target_categories = [
    'pdbx_audit_revision_category',
    'pdbx_audit_revision_details', 
    'pdbx_audit_revision_group',
    'pdbx_audit_revision_history',
    'pdbx_audit_revision_item',
    'pdbx_database_related',
    'pdbx_database_status',
    'pdbx_distant_solvent_atoms',
    'pdbx_entity_nonpoly',
    'pdbx_entry_details',
    'pdbx_nonpoly_scheme',
    'pdbx_poly_seq_scheme',
    'pdbx_struct_assembly',
    'pdbx_struct_assembly_gen',
    'pdbx_struct_assembly_prop',
    'pdbx_struct_oper_list',
    'pdbx_struct_sheet_hbond',
    'pdbx_validate_symm_contact',
    'pdbx_validate_torsion',
    'pdbx_xplor_file',
    'entity'
]

print("="*80)
print("查找特定类别在CIF文件中的位置和内容")
print("="*80)

# 读取原始CIF文件内容
with open("1qkt.cif", "r") as f:
    cif_content = f.read()

lines = cif_content.split('\n')

# 查找每个目标类别
for category in target_categories:
    print(f"\n🔍 查找类别: {category}")
    print("-" * 50)
    
    # 在已解析的数据中检查
    if category in data_block:
        cat_data = data_block[category]
        print(f"✅ 在解析数据中找到: {category}")
        print(f"   包含字段: {list(cat_data.keys())}")
        
        # 显示第一个字段的数据样例
        if cat_data.keys():
            first_key = list(cat_data.keys())[0]
            sample_data = cat_data[first_key].as_array()
            print(f"   样例数据 ({first_key}): {sample_data[:3] if len(sample_data) > 3 else sample_data}")
    else:
        print(f"❌ 在解析数据中未找到: {category}")
    
    # 在原文件中搜索
    found_lines = []
    for i, line in enumerate(lines):
        if category in line and (line.strip().startswith('_' + category) or line.strip().startswith('loop_')):
            found_lines.append((i+1, line.strip()))
    
    if found_lines:
        print(f"📍 在原文件中的位置:")
        for line_num, line_content in found_lines[:5]:  # 只显示前5个匹配
            print(f"   第 {line_num} 行: {line_content}")
    else:
        print(f"📍 在原文件中未找到明确的 _{category} 标记")

print("\n" + "="*80)